In [1]:
!pip install kaleido==0.2.1

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import textwrap
import os
import plotly.express as px

OUT_DIR = "eda_images"
os.makedirs(OUT_DIR, exist_ok=True)
df = pd.read_csv("/content/drive/MyDrive/StackOverFlow/survey_cleaned_base.csv")
map_base = pd.read_csv(os.path.join(OUT_DIR, "04b_salary_by_country_for_map.csv"))

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.titlesize': 14
})

def wrap_index(series_or_df, width=35):
    series_or_df.index = [textwrap.fill(str(idx), width) for idx in series_or_df.index]
    return series_or_df

COUNTRY_SHORT = {
    "United States of America": "USA",
    "United Kingdom of Great Britain and Northern Ireland": "UK",
    "Russian Federation": "Russia",
    "Republic of Korea": "South Korea"
}

if "median_salary" in map_base.columns:
    top30 = map_base.sort_values("respondents", ascending=False).head(30)
    top30 = top30.sort_values("median_salary", ascending=True)
    top30["display_name"] = top30["Country"].apply(lambda x: COUNTRY_SHORT.get(x, x))

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.barh(top30["display_name"], top30["median_salary"], color="#5B9BD5")
    ax.set_xlabel("Медианная зарплата (USD)")
    ax.set_title("Медианный годовой доход по странам")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "04_median_salary_by_country.png"), dpi=300)
    plt.close(fig)

    map_data = map_base.dropna(subset=["iso3"])
    map_data = map_data[map_data["respondents"] >= 30]
    cmax = np.quantile(map_data["median_salary"], 0.95)

    fig_map = px.choropleth(
        map_data, locations="iso3", color="median_salary", hover_name="Country",
        color_continuous_scale="Blues", range_color=(0, cmax), projection="natural earth"
    )
    fig_map.update_layout(title=dict(text="Географическое распределение медианной годовой зарплаты", x=0.5), margin=dict(l=0, r=0, t=60, b=0))
    fig_map.write_image(os.path.join(OUT_DIR, "04b_salary_map.png"), width=1600, height=900, scale=2)

if "wexp_bin" in df.columns and "salary_filtered" in df.columns:
    wexp_sal = df.groupby("wexp_bin", observed=True)["salary_filtered"].median()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(wwe_idx := wexp_sal.index.astype(str), wexp_sal.values, color="#ED7D31")
    ax.set_xlabel("Рабочий стаж (лет)")
    ax.set_ylabel("Медианная зарплата (USD)")
    ax.set_title("Медианный доход по группам профессионального стажу")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT_DIR, "09b_salary_by_workexp.png"), dpi=300)
    plt.close(fig)

for col in ["LanguageHaveWorkedWith", "DatabaseHaveWorkedWith"]:
    if col in df.columns and "salary_filtered" in df.columns:
        mask = df["salary_filtered"].notna() & df[col].notna()
        rows = []
        for _, r in df[mask].iterrows():
            for item in str(r[col]).split(";"):
                rows.append({"tech": item.strip(), "salary": r["salary_filtered"]})
        tdf = pd.DataFrame(rows)
        agg = tdf.groupby("tech")["salary"].agg(["median", "count"])

        top_med = agg[agg["count"] >= 50].sort_values("median", ascending=False).head(20)
        top_med = wrap_index(top_med, width=25)

        fig, ax = plt.subplots(figsize=(10, 6))
        top_med.sort_values("median").plot.barh(ax=ax, y="median", legend=False, color="#5B9BD5")
        ax.set_xlabel("Медианная зарплата (USD)")
        ax.set_title(f"Медианный доход по {col.split('Have')[0].lower()} (топ-20)")
        fig.tight_layout()
        fig.savefig(os.path.join(OUT_DIR, f"08_salary_by_{col}_top20.png"), dpi=300)
        plt.close(fig)

        pop = df[col].dropna().str.split(";").explode().str.strip().value_counts()
        merged = agg.join(pop.rename("popularity"), how="inner")
        merged = merged[merged["count"] >= 50].sort_values("popularity", ascending=False).head(25)

        sizes = (np.sqrt(merged["count"]) / np.sqrt(merged["count"]).max()) * 1800 + 100
        fig, ax = plt.subplots(figsize=(13, 8))
        sc = ax.scatter(merged["popularity"], merged["median"], s=sizes, alpha=0.55, c=merged["median"], cmap="viridis", edgecolor="black")
        for tech, row in merged.iterrows():
            ax.annotate(tech, (row["popularity"], row["median"]), fontsize=9, ha="center", va="center", fontweight="bold")
        ax.set_xlabel("Популярность (число респондентов)")
        ax.set_ylabel("Медианная зарплата (USD)")
        ax.set_title(f"{col.split('Have')[0]}: совместное распределение популярности и дохода")
        ax.grid(True, alpha=0.3)
        plt.colorbar(sc, ax=ax).set_label("Медианная зарплата (USD)")
        fig.tight_layout()
        fig.savefig(os.path.join(OUT_DIR, f"10_bubble_{col}.png"), dpi=300)
        plt.close(fig)